# SIMT 线程架构

## 概述

上一节我们了解了 SIMT 的抽象硬件架构——AI 处理器里有多个 Vector Core，每个 Vector Core 能并发执行上千个线程。本节要解决的核心问题是：**这么多线程，如何用同一份代码完成一次完整的计算？**

答案是 SIMT 的关键思想：**所有线程执行完全相同的一份代码，但每个线程通过内置变量算出不同的编号，从而各自处理不同的数据**。理解这一点，就抓住了 SIMT 编程的精髓。

### 前置要求

- 已理解 SIMT 抽象硬件架构中 AI 处理器、Vector Core 和存储资源的关系。
- 了解一维、二维、三维索引的基本含义，能够阅读简单的整数表达式。
- 本小节为理论讲解，不依赖在线硬件环境。

### 学习目标

学完本小节后，你应该能够：

- 解释 SIMT 编程模型中的 Grid、Thread Block 和 Thread 三层关系。
- 说清楚「所有线程共用一套代码、靠内置变量处理不同数据」是如何实现的。
- 使用 `blockIdx`、`blockDim`、`threadIdx` 计算每个线程负责的数据位置。
- 说明 Warp 的含义，理解 SIMT 在底层是如何调度线程的。

### 小节内容

- SIMT 线程层次结构
- 一套代码如何处理不同数据
- 线程索引如何计算
- Grid 和 Thread Block 的规模与限制
- Warp 与底层调度

### SIMT 线程层次结构

一个 SIMT 算子执行时通常会启动大量线程。为了便于把这些线程映射到硬件资源上，SIMT 编程模型采用分层的线程组织结构，从顶层到底层依次为：**Grid（线程块网格）、Thread Block（线程块）和 Thread（线程）**。

![SIMT线程层次结构](images/simt_thread_hierarchy.svg)

> 图中每个向下的绿色箭头代表一个线程（箭头数量只是示意，并不代表实际线程数量），Thread Block 调度到 Vector Core 的顺序由硬件决定，图中只是示意，不代表实际调度顺序。

三个层次的含义：

- **Thread（线程）**：最小的执行单元。每个线程独立完成自己的计算任务，拥有独立的寄存器。
- **Thread Block（线程块）**：由若干线程组成。一线程块会被调度到一个vector core上执行。
- **Grid（线程块网格）**：最顶层，由多个结构相同的线程块组成。

一次 SIMT 算子执行可以理解为：启动一个 **Grid**，Grid 里有多个结构相同的 **Thread Block**，每个 Thread Block 又包含多个 **Thread**。这些Thread Block会被调度到物理的vector core上执行，等 Grid 内所有 Thread Block 都调度执行完后，这个算子就执行完了。

这套分层结构的意义在于：开发者只需描述线程的组织方式，硬件就能把大量线程合理地映射到多个 Vector Core 上并行执行。而在 Vector Core 内部，线程块内的线程还会以 32 个为一组（一个 **Warp**）来调度，Warp 的细节会在本节最后展开。

### 一套代码如何处理不同数据

SIMT 算子执行时，**所有线程执行的是同一份代码**，但每个线程处理的是**不同的数据**，从而实现并行计算。这正是 SIMT（Single Instruction, Multiple Threads，单指令多线程）名字的由来。

那么问题来了：既然代码完全一样，每个线程怎么知道自己该处理哪一份数据？

答案是：**靠内置变量**。每个线程在运行时都能读到几个由硬件自动赋值的内置变量，从而算出一个专属于自己的全局编号，再用这个编号去取对应的数据。常用的内置变量有：

| 内置变量 | 含义 |
| --- | --- |
| `blockIdx` | 当前线程所在的线程块，在 Grid 中的索引 |
| `threadIdx` | 当前线程在所属线程块内部的索引 |
| `blockDim` | 一个线程块内的线程数量 |
| `gridDim` | 一个Grid内线程块数量 |

下面以**向量加法** `C[i] = A[i] + B[i]` 为例。假设每个线程块有 4 个线程（`blockDim.x = 4`），启动 2 个线程块，一共 8 个线程，正好处理一个长度为 8 的数组：

![线程索引映射示例](images/thread_index_mapping.svg)

可以看到，8 个线程跑的是同一行计算代码 `C[i] = A[i] + B[i]`，区别只在于每个线程算出来的 `i` 不同：

- Block 0 的 4 个线程算出 `i = 0,1,2,3`；
- Block 1 的 4 个线程算出 `i = 4,5,6,7`。

于是 8 个线程各自处理一个数组元素，互不重叠，合在一起就完成了整个向量的加法。这就是「一套代码 + 内置变量」实现数据并行的核心机制。

### 线程索引如何计算

上一节图里每个线程的全局编号 `i`，正是用内置变量算出来的。最常用的一维计算公式是：

```text
i = blockIdx.x * blockDim.x + threadIdx.x
```

这个公式可以拆成两步理解：

1. `blockIdx.x * blockDim.x`：先跳过前面所有线程块占用的编号。比如 Block 1（`blockIdx.x = 1`），每个线程块有 4 个线程，那么前面已经有 `1 * 4 = 4` 个线程，因此编号从 4 开始。
2. `+ threadIdx.x`：再加上当前线程在本线程块内的偏移，得到全局编号。

代入验证：Block 1 内 `threadIdx.x = 2` 的线程，`i = 1 * 4 + 2 = 6`，正好对应数组元素 `A[6]`，与上图一致。

#### 扩展到三维：dim3

实际编程中，数据不一定是一维数组，也可能是二维图像、三维张量。因此 `gridDim`、`blockDim`、`blockIdx`和`threadIdx` 都用一个三维结构 `dim3` 表示，可以理解为 `{x, y, z}` 三个方向上的规模：

- `gridDim`：Grid 在 3 个维度上各有多少个线程块，线程块用 `blockIdx` 定位。
- `blockDim`：每个线程块在 3 个维度上各有多少个线程，线程用 `threadIdx` 定位。

一维只是三维的特例（y、z 方向都为 1）。初学阶段先把一维的 `i = blockIdx.x * blockDim.x + threadIdx.x` 理解透，二维、三维只是在每个方向上重复同样的思路。

### Grid 和 Thread Block 的规模与限制

知道了线程怎么定位数据，再来看启动多少线程是有约束的。Grid 和 Thread Block 的规模分别由内置变量 `gridDim` 和 `blockDim` 描述，并各有限制：

| 层次 | 含义 | 描述规模的变量 | 定位用的变量 | 关键限制 |
| --- | --- | --- | --- | --- |
| Grid | 由多个 Thread Block 组成 | `gridDim` | `blockIdx` | 三维乘积总数不超过 65535 |
| Thread Block | 由若干 Thread 组成 | `blockDim` | `threadIdx` | 三维乘积总数不超过 1024（默认，可调整） |

Grid 和 Thread Block 的规模如何配置将在下一章节介绍。总线程数计算方式如下：

```text
total_threads = (gridDim.x * gridDim.y * gridDim.z) * (blockDim.x * blockDim.y * blockDim.z)
```

关于线程块之间的关系，有几点需要了解：

- 同一个 Grid 中的所有线程块**尺寸和维度配置相同**。
- 同一个 Grid 中的线程块**相互独立，可以按任意顺序执行**，彼此之间不存在固定的先后关系。

正因为线程块相互独立，硬件才能灵活地把它们分配到多个 Vector Core 上并行执行，这也是 SIMT 能扩展到大规模并行的基础。

### Warp 与底层调度

前面都是从开发者视角看线程：我们组织的是 Grid、Thread Block 和 Thread。但从硬件**底层执行**的视角看，一个线程块内的线程其实**无法同时全部并发执行**，而是被进一步划分成多个 **Warp** 来调度。

关键规则如下：

- 在一个线程块内，所有线程按线性顺序被硬件自动划分为每 **32 个线程一组**的 Warp。
- Warp 是 SIMT 架构中基本的调度和执行单位，**同一个 Warp 内的所有线程执行同一条指令**。
- Warp 内的线程从相同的程序地址开始执行，但各自拥有独立的寄存器状态，可以走不同的分支。

**分支发散（Warp Divergence）**：Warp 内的线程虽然执行同一段代码，但可能因为条件分支进入不同的执行路径，这种情况称为分支发散。当 32 个线程都走相同分支时，硬件利用率最高；一旦发生分支发散，硬件会串行执行每条分支路径，只有进入当前分支的线程会被执行，其余线程被屏蔽，从而降低执行效率。

**理解 SIMT 的真正含义**：到这里就能回答「为什么所有线程能共用一套代码」了。硬件以 Warp 为单位，让一组线程执行同一条指令——这就是 **Single Instruction（单指令）**；而每个线程靠 `threadIdx` 等内置变量访问各自的数据——这就是 **Multiple Threads（多线程）**。一条指令、多个线程、各取各的数据，正是 SIMT 的精髓。

并且基于这一点，有一条实用建议：**线程块内的线程总数建议设置为 32 的整数倍**。否则最后一个 Warp 不足 32 个线程，会存在空闲的线程通道，降低执行效率。

### 术语速查

| 术语 | 说明 |
| --- | --- |
| Grid | 一次算子启动时的全部线程块集合，是线程层次的最顶层。 |
| Thread Block | 由若干线程组成的一组线程，块内线程可通过共享内存交互、通过同步机制协作。 |
| Thread | SIMT 程序中的基本执行单元，所有线程执行同一份代码。 |
| `gridDim` | Grid 的三维规模，表示启用的线程块个数。 |
| `blockDim` | Thread Block 的三维规模，表示块内启用的线程个数。 |
| `blockIdx` | 当前线程块在 Grid 中的索引。 |
| `threadIdx` | 当前线程在所属线程块内的索引。 |
| Warp | SIMT 基本的调度和执行单位，32 个线程一组，组内执行同一条指令。 |
| Warp Divergence | Warp 内线程因条件分支走不同路径，硬件需串行执行各分支，降低效率。 |

## 小节小结

本小节的主线是：**SIMT 如何用一套代码完成大规模数据计算**。

- **线程层次**：一次算子执行启动一个 Grid，Grid 包含多个 Thread Block，每个 Thread Block 包含多个 Thread。
- **核心机制**：所有线程执行同一份代码，但每个线程用内置变量 `blockIdx`、`blockDim`、`threadIdx` 算出专属的全局编号，再用编号各取各的数据，互不重叠。
- **底层调度**：线程块被硬件按线性顺序划分成多个 Warp（每组 32 线程），Warp 是基本的调度和执行单位，组内线程执行同一条指令。因此 `blockDim` 建议设为 32 的整数倍，并尽量避免分支发散。

理解了「同一套代码 + 内置变量定位数据」这个机制，就为学习核函数编写打下了基础。下一章节《核函数》会带你把这套线程模型真正写成可执行的算子代码。

## 课后练习

本节讲解了 SIMT 的线程层次结构，以及「所有线程共用一套代码、靠内置变量处理不同数据」的核心机制，请根据学习内容完成以下题目进行自测。

1. （判断题）SIMT 算子执行时，不同线程执行的是各自不同的代码，因此才能处理不同的数据。

2. （单选题）SIMT 线程层次结构从顶层到底层依次是？  
    A. Thread → Thread Block → Grid  
    B. Grid → Thread Block → Thread  
    C. Grid → Warp → Thread  
    D. Thread Block → Grid → Thread  

3. （单选题）已知 `blockDim.x = 4`，对于 Block 1 内 `threadIdx.x = 2` 的线程，按 `global_id = blockIdx.x * blockDim.x + threadIdx.x` 计算，它的全局索引是多少？  
    A. 2  
    B. 4  
    C. 6  
    D. 8  

4. （多选题）以下关于「所有线程共用一套代码」和 Warp 执行机制的说法，哪些是正确的？  
    A. 所有线程执行完全相同的一份代码，每个线程通过内置变量算出不同的全局编号  
    B. 同一 Warp 由 32 个线程组成，且执行同一条指令  
    C. 每个线程执行的是各自不同的代码，因此才能处理不同数据  
    D. 当 Warp 内线程因条件分支走不同路径（分支发散）时，硬件会串行执行各分支，降低效率  

**执行以下代码获取答案。**

## 参考答案

运行下面的单元格查看课后练习参考答案。


In [ ]:
!cat answer/03.04.02_answer.txt
